In [1]:
import calendar
import json
import os
import pickle
import random
import re
import sys
from datetime import date
from typing import List

import dateutil
import graphistry
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import DataFrame, SparkSession

In [2]:
GRAPHISTRY_KEY_ID = os.environ["GRAPHISTRY_KEY_ID"]
GRAPHISTRY_API_KEY = os.environ["GRAPHISTRY_API_KEY"]

In [3]:
# Configuration for Graphistry
GRAPHISTRY_PARAMS = {
    "play": 500,
    "pointOpacity": 0.7,
    "edgeOpacity": 0.3,
    "edgeCurvature": 0.3,
    "showArrows": True,
    "gravity": 0.15,
    "showPointsOfInterestLabel": False,
    "labels": {
        "shortenLabels": False,
    },
}
FAVICON_URL = "https://graphlet.ai/assets/icons/favicon.ico"
LOGO = {"url": "https://graphlet.ai/assets/Branding/Graphlet%20AI.svg", "dimensions": {"maxWidth": 100, "maxHeight": 100}}

In [4]:
# Initialize a SparkSession
spark: SparkSession = (
    SparkSession.builder.appName("Stack Overflow Pregel API")
    # Lets the Id:(Stack Overflow int) and id:(GraphFrames ULID) coexist
    .config("spark.sql.caseSensitive", True)
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir("/tmp/graphframes-checkpoints")

25/04/18 12:30:53 WARN Utils: Your hostname, Achilles.local resolves to a loopback address: 127.0.0.1; using 10.0.0.246 instead (on interface en0)
25/04/18 12:30:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/18 12:30:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
!ls ../data/knowledge_graph

companies.parquet                    products.parquet
company_ticker_relationships.parquet tech_company_relationships.parquet
documents.parquet                    technologies.parquet
product_company_relationships        tickers.parquet


In [17]:
company_df = spark.read.parquet("../data/knowledge_graph/companies.parquet")
print(f"Company Count: {company_df.count():,}")
company_df.show(5)

Company Count: 354
+----+--------------------+---------+------------+---------------------+------------+-----------------+-----------+------+-----------+
| ceo|         description|employees|founded_year|headquarters_location|linkedin_url|             name|revenue_usd|ticker|website_url|
+----+--------------------+---------+------------+---------------------+------------+-----------------+-----------+------+-----------+
|NULL|ACM Research is a...|     NULL|        1998|           California|        NULL|     ACM Research|  250000000|  NULL|       NULL|
|NULL|AMD is a semicond...|     NULL|        NULL|                 NULL|        NULL|              AMD|       NULL|  NULL|       NULL|
|NULL|Shanghai-based co...|     NULL|        NULL|                 NULL|        NULL|             AMEC|       NULL|  NULL|       NULL|
|NULL|Purchased lithogr...|     NULL|        NULL|                 NULL|        NULL|              ASE|       NULL|  NULL|       NULL|
|NULL|A major toolmaker...|     NULL

In [16]:
product_df = spark.read.parquet("../data/knowledge_graph/products.parquet")
print(f"Product Count: {product_df.count():,}")
product_df.show(5)

Product Count: 411
+--------------------+--------------------+--------------------+------------+
|             company|         description|                name|company_name|
+--------------------+--------------------+--------------------+------------+
|{NULL, Taiwan Sem...|A semiconductor m...|         3nm FinFlex|        TSMC|
|{NULL, Qualcomm i...|A PCIe inline acc...|5G Distributed Un...|    Qualcomm|
|{NULL, Qualcomm i...|Entire solutions ...|5G front end modules|    Qualcomm|
|{NULL, The larges...|Capabilities for ...|800G optical tran...|       Cisco|
|{NULL, Nvidia is ...|         Nvidia GPU.|                A100|      Nvidia|
+--------------------+--------------------+--------------------+------------+
only showing top 5 rows



In [14]:
technology_df = spark.read.parquet("../data/knowledge_graph/technologies.parquet")
print(f"Technology Count: {technology_df.count():,}")
technology_df.show(5)

Technology Count: 356
+--------------------+--------------------+-----------+---------------+
|         description|           developer|       name| developer_name|
+--------------------+--------------------+-----------+---------------+
|Process node orig...|{NULL, Intel is t...|       10nm|          Intel|
|The most advanced...|{NULL, foundry, N...|       12nm|GlobalFoundries|
|Process node wher...|{NULL, Intel is t...|       14nm|          Intel|
|Almost all equipm...|{NULL, China’s la...|14nm FinFET|           SMIC|
|Semiconductor man...|{NULL, Taiwan Sem...|       16nm|           TSMC|
+--------------------+--------------------+-----------+---------------+
only showing top 5 rows



In [13]:
ticker_df = spark.read.parquet("../data/knowledge_graph/tickers.parquet")
print(f"Ticker Count: {ticker_df.count():,}")
ticker_df.orderBy("name").show(10)

Ticker Count: 48
+--------+--------------------+------+
|exchange|                name|symbol|
+--------+--------------------+------+
|    NULL|         ASM Pacific| ASMPT|
|    NULL|                ASML|  ASML|
|    NULL|Advanced Micro De...|   AMD|
|      JP|           Advantest|  6857|
|    NULL|   Aehr Test Systems|  AEHR|
|    NULL|             Aixtron|  AIXA|
|    NULL|             Alibaba|  BABA|
|    NULL|       Alphabet Inc.| GOOGL|
|       L|      Alphawave Semi|   AWE|
|    NULL|              Amazon|  AMZN|
+--------+--------------------+------+
only showing top 10 rows



In [20]:
doc_df = spark.read.parquet("../data/knowledge_graph/documents.parquet")
print(f"Total Documents: {doc_df.count():,}")
doc_df.show(5)

Total Documents: 171
+--------------------+------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+
|             authors|collected_at|           companies|            products|published_at|             summary|        technologies|             tickers|               title|
+--------------------+------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+
|[{SemiAnalysis, N...|        NULL|[{NULL, Nvidia is...|[{{NULL, Nvidia i...|        NULL|Nvidia was hacked...|[{Technology that...|[{NULL, Nvidia, N...|Nvidia was hacked...|
|                NULL|        NULL|[{NULL, China’s a...|[{{NULL, China’s ...|        NULL|Biren, China’s ar...|                  []|                NULL|The regulations a...|
|                NULL|        NULL|                  []|                  []|        NULL|The article di